# Web IQ: REST API 직접 호출

## 목표와 사전 준비

포털 밖의 Python에서 Microsoft Foundry와 Microsoft Copilot을 Web, Images, Videos로 검색합니다. 세 Vertical에 같은 검색어를 그대로 전달하며 `YouTube`를 자동으로 덧붙이지 않습니다. **고객 실습에 사용이 허용된 진행자 제공 키로 참가자가 직접 실행**합니다. 포털 접근·키 발급은 참가자에게 전제하지 않으며, 키가 없으면 [설명 자료](README.md)의 기존 캡처를 분석합니다.

- Python 3.11 이상, VS Code Jupyter 확장, 인터넷 연결이 필요합니다.
- 진행자가 [Profile Management](https://webiq.microsoft.ai/profiles)에서 발급해 제공한 실습 키를 `WEBIQ_API_KEY` 환경 변수 또는 숨김 입력으로 전달합니다. 키를 코드에 붙여 넣지 않습니다.
- 실행 위치는 이 노트북입니다. 위에서 아래로 실행하며, 검색 셀은 6회 API 호출을 사용합니다. 계정별 허용 서비스·할당량·요금을 먼저 확인하세요.
- REST는 `POST https://api.microsoft.ai/v3/search/{vertical}`와 `x-apikey` 헤더를 사용합니다. LLM 호출이나 Azure 리소스 배포는 필요하지 않습니다.

공식 문서: [Web](https://webiq.microsoft.ai/documentation/api-reference/web/) · [Images](https://webiq.microsoft.ai/documentation/api-reference/images/) · [Videos](https://webiq.microsoft.ai/documentation/api-reference/videos/) · [인증](https://webiq.microsoft.ai/documentation/authentication/)

In [1]:
%pip install "httpx>=0.28,<1"

Looking in indexes: https://packagefeedproxy.microsoft.io/pypi/simple/



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /Users/changjuahn/Repo/MicrosoftIQSeries-KR/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import getpass
import json
import os
from datetime import datetime, timezone

import httpx

api_key = (os.environ.get("WEBIQ_API_KEY") or getpass.getpass("Web IQ API key: ")).strip()
if not api_key:
    raise ValueError("Web IQ API 키가 필요합니다.")
print("API 키 준비 완료. 키 값은 출력하지 않습니다.")
print("실행 시각 (UTC):", datetime.now(timezone.utc).isoformat())

API 키 준비 완료. 키 값은 출력하지 않습니다.
실행 시각 (UTC): 2026-09-30T01:33:58.922189+00:00


In [ ]:
results = {}
result_fields = {"web": "webResults", "images": "imageResults", "videos": "videoResults"}

with httpx.Client(
    base_url="https://api.microsoft.ai",
    headers={"x-apikey": api_key},
    timeout=60.0,
) as client:
    for topic in ("Microsoft Foundry", "Microsoft Copilot"):
        results[topic] = {}
        for vertical, result_field in result_fields.items():
            payload = {"query": topic, "maxResults": 3, "language": "en", "region": "US"}
            if vertical == "web":
                payload.update(contentFormat="passage", maxLength=800)

            endpoint = f"/v3/search/{vertical}"
            response = client.post(endpoint, json=payload)
            print(f"\nPOST https://api.microsoft.ai{endpoint} | {topic} | HTTP {response.status_code}")
            if response.status_code in (401, 403, 429):
                reasons = {401: "키가 유효한지 확인하세요.", 403: "계정의 서비스 권한을 확인하세요.", 429: "할당량과 호출 빈도를 확인하세요."}
                raise RuntimeError(f"HTTP {response.status_code}: {reasons[response.status_code]}")
            response.raise_for_status()
            result = response.json()
            items = result.get(result_field, [])
            if not items:
                raise RuntimeError(f"{topic} / {vertical}: 반환 결과가 없습니다. 검색어와 응답을 확인하세요.")
            results[topic][vertical] = result
            print(f"{result_field}: {len(items)}건 | traceId: {result.get('traceId', '(없음)')}")
            for item in items:
                print("제목:", item.get("title", "(없음)"))
                print("URL:", item.get("url", "(없음)"))
                print("내용:", (item.get("content") or item.get("caption") or item.get("description") or "")[:220])
                if vertical == "images":
                    print("원문 페이지:", item.get("hostPageUrl", "(없음)"))
                if vertical == "videos":
                    print("게시자:", item.get("publishedBy", "(없음)"))

## 결과 해석과 정리

- `webResults`: 제목, 출처 URL, `content`를 확인합니다. `passage`는 검색어에 관련된 문맥을 추출합니다. REST Web에는 `snippet` 필드가 없습니다.
- `imageResults`: 이미지 URL과 `hostPageUrl`은 다릅니다. `caption`은 생성된 설명이며 이미지의 사용 권리를 보장하지 않습니다.
- `videoResults`: 다른 Vertical과 같은 검색어를 사용하며 YouTube 전용 검색이 아닙니다. `publishedBy`, 실제 `url`, `length`를 확인합니다. `summary`, `moments` 등은 없을 수 있습니다.
- 검색 순위·본문·반환 건수는 실행 시점과 권한에 따라 달라집니다. 검색 성공이 원문의 정확성이나 접근 가능성을 보장하지 않습니다.
- 401은 키, 403은 서비스 권한, 429는 할당량을 확인합니다. 그 외 HTTP 오류는 상태 코드로 확인하며 자동 재시도하지 않습니다.

아래 셀은 첫 웹 결과의 일부를 원래 필드명으로 출력합니다. 전체 원문 대신 짧은 발췌만 공유합니다. 이후 MCP 실습까지 마치면 **이번에 만든 데모 키만** Profile Management에서 폐기하세요. 기존 키는 삭제하지 않습니다. 커널을 재시작해 메모리의 키·클라이언트·응답도 정리합니다.

In [ ]:
first_result = results["Microsoft Foundry"]["web"]["webResults"][0]
print(json.dumps({
    "title": first_result.get("title"),
    "url": first_result.get("url"),
    "content_excerpt": first_result.get("content", "")[:220],
}, ensure_ascii=False, indent=2))
del api_key
print("키 변수를 제거했습니다. 실습 종료 후 커널을 재시작하고 데모 전용 키를 폐기하세요.")